# Train metrics analysis: baseline G=8 vs ablation G=4

Load both runs' `train_metrics.jsonl`, compute summary statistics, and plot per-metric trajectories side-by-side. The load-bearing question for the writeup: how much of the eval-accuracy gap is explained by G=4 spending more update steps on zero-advantage prompts?

In [ ]:
# Setup environment in colab
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
%cd /content/drive/MyDrive/grpo-qwen0.5/
!git pull

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
RESULTS = Path("results")

runs = {
    "G=8 (baseline)": sorted(RESULTS.glob("baseline-*"))[-1],
    "G=4 (ablation)": sorted(RESULTS.glob("ablation_g4-*"))[-1],
}

frames = {}
for label, run_dir in runs.items():
    df = pd.read_json(run_dir / "train_metrics.jsonl", lines=True)
    frames[label] = df
    print(f"{label}: {run_dir.name}  ->  {len(df)} log rows")

In [ ]:
metrics = ["zero_adv_fraction", "reward_mean", "reward_std",
           "grad_norm", "kl", "mean_completion_len", "loss"]

rows = []
for label, df in frames.items():
    for m in metrics:
        rows.append({
            "run": label,
            "metric": m,
            "mean": df[m].mean(),
            "std": df[m].std(),
            "first5_mean": df[m].head(5).mean(),
            "last5_mean": df[m].tail(5).mean(),
        })

summary = pd.DataFrame(rows).pivot(index="metric", columns="run",
                                    values=["mean", "std"])
summary.round(4)

### Steady-state stats (drop step-10 init transient)

The very first logged step (step 10) carries an initialization-transient gradient that dominates `grad_norm` std with only 50 logged points. Recomputing without it gives the steady-state picture used in the writeup.

In [ ]:
rows_ss = []
for label, df in frames.items():
    df_ss = df.iloc[1:]
    for m in metrics:
        rows_ss.append({
            "run": label,
            "metric": m,
            "mean": df_ss[m].mean(),
            "std": df_ss[m].std(),
            "first5_mean": df_ss[m].head(5).mean(),
            "last5_mean": df_ss[m].tail(5).mean(),
        })

summary_ss = pd.DataFrame(rows_ss).pivot(index="metric", columns="run",
                                          values=["mean", "std"])
summary_ss.round(4)

## Per-metric trajectories

Six metrics, two runs each. Each panel is on its own scale - read shape, not magnitude across panels.

In [ ]:
plot_metrics = ["zero_adv_fraction", "reward_mean", "reward_std",
                "grad_norm", "kl", "mean_completion_len"]

fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True)

for ax, metric in zip(axes.flat, plot_metrics):
    for label, df in frames.items():
        ax.plot(df["step"], df[metric], marker="o", markersize=3,
                linewidth=1, label=label, alpha=0.85)
    ax.set_title(metric)
    ax.set_xlabel("step")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

fig.suptitle("Train-time metrics: G=8 vs G=4", fontsize=13)
fig.tight_layout()
fig.savefig(RESULTS / "train_metrics_comparison.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
print("=== Load-bearing mechanism numbers ===\n")
for label, df in frames.items():
    z = df["zero_adv_fraction"].mean()
    rs = df["reward_std"].mean()
    print(f"{label:18s}  zero_adv_fraction = {z:.3f}   reward_std = {rs:.3f}")

z8 = frames["G=8 (baseline)"]["zero_adv_fraction"].mean()
z4 = frames["G=4 (ablation)"]["zero_adv_fraction"].mean()
print(f"\nG=4 wastes {z4/z8:.2f}x as many micro-batches on zero-advantage prompts.")
print(f"Effective gradient signal per wall-clock hour, G=4 vs G=8: ~{(1-z4)/(1-z8):.2f}x")

## Findings

1. **`zero_adv_fraction` is structurally higher and stable for G=4** (~48% vs ~30%). It does not climb during training - this is a property of group size, not policy saturation.
2. **`grad_norm` (steady-state, drop step 10): G=8 mean 2.16 / std 0.16; G=4 mean 2.32 / std 0.22**. G=4's grad_norm std is ~40% higher in steady state - consistent with the prediction that smaller G yields noisier per-prompt advantage estimates. The full-run std is dominated by an init-transient spike at step 10 (G=8 ~8.1, G=4 ~3.7) which is not reflective of training behavior.
3. **`kl` stays small and grows healthily in both runs** (no KL blowup); beta=0.04 is not pinning the policy.
4. **`mean_completion_len` drifts ~322 -> ~305 in both runs**, identically - the runs are not differentiated by response-length artifacts.
5. **Reward floor**: G=8's `reward_mean` crosses zero around step 200 and stays positive; G=4's stays slightly negative throughout, consistent with its lower eval ceiling.